# Overfitting and Model Selection Practice

**Topic:** Model Training and Generalization &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `x_train` `(60,)`, `y_train`: a noisy smooth function on [0, 3], deliberately small.
- `x_val` `(60,)`, `y_val`: a validation set you may use freely for selection.
- `x_test` `(300,)`: predict `y` here. `y_test` is used by the evaluation cell.

## Your task

1. Build polynomial features of `x` for several degrees (e.g., 1 through 15). `np.vander` or `PolynomialFeatures` both work, but **rescale `x` to [0, 1] first** (divide by 3) so high powers stay numerically sane, and a ridge with small alpha helps too.
2. Pick the degree (and alpha) with the best **validation** error. Plot train vs validation error against degree if you want to *see* overfitting happen.
3. Refit your chosen model and set `predictions` to its outputs at `x_test`.

## Evaluation

Test R² must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

def truth(x):
    return 1.5 * np.sin(2.2 * x) + 0.5 * x

def make(n, seed, lo=0.0, hi=3.0):
    r = np.random.default_rng(seed)
    x = r.uniform(lo, hi, n)
    return x, truth(x) + r.normal(0, 0.25, n)

x_train, y_train = make(60, 121)
x_val, y_val = make(60, 122)
x_test, y_test = make(300, 123)
print(f"train {x_train.shape}, val {x_val.shape}, test {x_test.shape}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_R2 = 0.8

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions, dtype=float).ravel()
_true = np.asarray(y_test, dtype=float).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

_rmse = float(np.sqrt(np.mean((_pred - _true) ** 2)))
_ss_res = float(np.sum((_true - _pred) ** 2))
_ss_tot = float(np.sum((_true - _true.mean()) ** 2))
r2 = 1.0 - _ss_res / _ss_tot
print("=" * 52)
print(f"Test RMSE : {_rmse:.4f}")
print(f"Test R^2  : {r2:.4f}   (target: >= {TARGET_R2:.2f})")
print(f"Worst absolute error: {float(np.max(np.abs(_pred - _true))):.4f}")
print("=" * 52)
if r2 >= TARGET_R2:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your model and try again.")


## Solution

Sweep polynomial degree (on x rescaled to [0, 1]) with a small ridge penalty, score each candidate on the validation set, and keep the winner. The validation curve turns up exactly where overfitting begins.

```python
import numpy as np
from sklearn.linear_model import Ridge

def feats(x, degree):
    return np.vander(np.asarray(x) / 3.0, degree + 1, increasing=True)

best = None
for degree in range(1, 16):
    model = Ridge(alpha=1e-4).fit(feats(x_train, degree), y_train)
    val_mse = float(np.mean((model.predict(feats(x_val, degree)) - y_val) ** 2))
    if best is None or val_mse < best[1]:
        best = (degree, val_mse, model)

degree, val_mse, model = best
print(f"chosen degree {degree} (val MSE {val_mse:.4f})")
predictions = model.predict(feats(x_test, degree))
```
